# AI Product Mini-Projekt 

- Laden Finanz-Daten für beliebige Firmen über SEC-API 
- Berechne Bewertungsmultiples
- Stelle sie dar über eine Dash-GUI 
- Ermögliche das Stellen von Fragen über LLM
- Binde bei Beantwortung 10-K Filings über RAG ein


## Setup

In [19]:
# Falls noetig, einmalig ausfuehren:
# %pip install pandas requests plotly dash

import os
import pandas as pd
# Library zum Erstellen von Grafiken
import plotly.express as px
import plotly.graph_objects as go
# Library zum Erstellen von Web-Dashboards
from dash import Dash, dcc, html, Input, Output, State
# Library zum Abrufen von Finanzdaten
import requests

import requests
from getpass import getpass

import main_api

Die folgende Zelle enthält Utilities, die wir nicht besprechen werden. Unter anderen wird die folgende Funktion implementiert:

```python
def load_three_metrics(cik: int, start_year: int, end_year: int) -> pd.DataFrame

""" returns for a cik a panda dataframe with balancesheet information
"""
```

Bitte "klappen" Sie den Zellinhalt ein, ignorieren Sie ihren weiteren Inhalt, aber führen Sie die Zelle aus!


In [20]:
%%capture

# SEC Company Facts API helper

from __future__ import annotations

import requests

SEC_HEADERS = {
    "User-Agent": "teaching-dash/1.0 (contact@example.com)",
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov",
}

_BASE_URL = "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
_ANNUAL_FORMS = {"10-K", "10-K/A", "20-F", "20-F/A"}


def _normalize_cik(cik: int | str) -> str:
    return str(cik).strip().zfill(10)


def _latest_fy_value(companyfacts: dict, taxonomy: str, tag: str, fy: int, unit: str):
    entries = (
        companyfacts.get("facts", {})
        .get(taxonomy, {})
        .get(tag, {})
        .get("units", {})
        .get(unit, [])
    )
    if not entries:
        return None

    annual = [
        row
        for row in entries
        if row.get("fy") == fy
        and row.get("val") is not None
        and str(row.get("form", "")).upper() in _ANNUAL_FORMS
    ]
    if not annual:
        return None

    # Prefer most recently filed value for a given FY.
    annual.sort(key=lambda x: (x.get("filed", ""), x.get("end", "")))
    return annual[-1].get("val")


def get_core_metrics_from_companyfacts(cik: int | str, fy: int) -> dict:
    """Return core annual metrics for one company and fiscal year.

    Returns keys:
    - diluted_eps
    - capex_abs
    - operating_cf
    - free_cash_flow
    """
    cik10 = _normalize_cik(cik)
    url = _BASE_URL.format(cik=cik10)

    response = requests.get(url, headers=SEC_HEADERS, timeout=30)
    response.raise_for_status()
    companyfacts = response.json()

    diluted_eps = _latest_fy_value(companyfacts, "us-gaap", "EarningsPerShareDiluted", fy, "USD/shares")

    capex_raw = _latest_fy_value(
        companyfacts,
        "us-gaap",
        "PaymentsToAcquirePropertyPlantAndEquipment",
        fy,
        "USD",
    )
    # SEC cash flow outflows are often negative, but teaching examples expect positive CapEx.
    capex_abs = abs(capex_raw) if capex_raw is not None else None

    operating_cf = _latest_fy_value(
        companyfacts,
        "us-gaap",
        "NetCashProvidedByUsedInOperatingActivities",
        fy,
        "USD",
    )

    free_cash_flow = None
    if operating_cf is not None and capex_abs is not None:
        free_cash_flow = operating_cf - capex_abs

    return {
        "diluted_eps": diluted_eps,
        "capex_abs": capex_abs,
        "operating_cf": operating_cf,
        "free_cash_flow": free_cash_flow,
    }

def load_three_metrics(cik: int, start_year: int, end_year: int) -> pd.DataFrame:
    rows = []
    for fy in range(start_year, end_year + 1):
        core = get_core_metrics_from_companyfacts(cik=cik, fy=fy)

        diluted_earnings = core.get("diluted_eps")
        capex = core.get("capex_abs")
        ocf = core.get("operating_cf")
        fcf = core.get("free_cash_flow")

        if fcf is None and ocf is not None and capex is not None:
            fcf = ocf - capex
            fcf_mode = "calculated_ocf_minus_capex"
        else:
            fcf_mode = "direct_from_companyfacts"

        rows.extend([
            {"year": fy, "metric": "diluted_earnings", "value": diluted_earnings, "form": "10-K", "mode": "direct"},
            {"year": fy, "metric": "capex", "value": capex, "form": "10-K", "mode": "normalized_abs"},
            {"year": fy, "metric": "fcf", "value": fcf, "form": "10-K", "mode": fcf_mode},
        ])

    df = pd.DataFrame(rows)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df


# Helper Fkt. - nicht wichtig zum Verständnis!
def _build_openrouter_url() -> str:
    """Akzeptiert verschiedene Base-URL-Formate und erzeugt immer den korrekten Endpoint."""
    raw = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1/chat/completions").strip()
    base = raw.rstrip("/")

    if base.endswith("/chat/completions"):
        return base
    if base.endswith("/api/v1"):
        return f"{base}/chat/completions"
    if base == "https://openrouter.ai":
        return "https://openrouter.ai/api/v1/chat/completions"
    if "openrouter.ai" in base and "/api/v1" not in base:
        return f"{base}/api/v1/chat/completions"
    return base



"""
download_filings.py

Werkzeuge zum Herunterladen von SEC 10-K Filings als HTML.
Verwendet die SEC EDGAR API um Accession Numbers zu finden und
die entsprechenden Filing-Dokumente herunterzuladen.
"""

import requests
import json
import os
import time
from typing import Optional, Dict, List
from pathlib import Path
from dataclasses import asdict

# ============================================================================
# ARCHITEKTUR-ÜBERSICHT: FilingDownloader Klasse
# ============================================================================
#
# USER
#   │
#   └─> FilingDownloader.__init__(download_dir, user_agent)
#           └─> Erstellt download_dir
#           └─> Setzt SEC API headers
#
# ┌─────────────────────────────────────────────────────────────────────────┐
# │ HAUPTMETHODE FÜR CHATBOT (All-in-One)                                   │
# └─────────────────────────────────────────────────────────────────────────┘
#
# download_and_extract_for_chatbot(cik, current_year, company_name)
#     │
#     ├─> download_complete_package(cik, current_year, company_name)
#     │       │
#     │       ├─> download_multiple_years(cik, [year-2, year-1, year])
#     │       │       │
#     │       │       └─> download_10k_for_year(cik, year) [×3]
#     │       │               │
#     │       │               ├─> find_10k_filing(cik, year)
#     │       │               │       └─> get_submissions(cik)
#     │       │               │               └─> SEC API: submissions/CIK.json
#     │       │               │
#     │       │               ├─> download_filing_html(primary_doc)
#     │       │               │       └─> SEC API: edgar/data/.../primary.html
#     │       │               │
#     │       │               ├─> download_complete_submission(accession)
#     │       │               │       └─> SEC API: edgar/data/.../accession.txt
#     │       │               │
#     │       │               └─> download_filing_index(accession)
#     │       │                       └─> SEC API: edgar/data/.../-index.html
#     │       │
#     │       └─> download_latest_10q(cik, company_name)
#     │               │
#     │               ├─> find_latest_10q_filing(cik)
#     │               │       └─> get_submissions(cik)
#     │               │
#     │               ├─> download_filing_html(primary_doc)
#     │               ├─> download_complete_submission(accession)
#     │               └─> download_filing_index(accession)
#     │
#     └─> extract_clean_10k_text(filing_paths) [×4 Filings]
#             │
#             ├─> extract_10k_text_from_complete_submission(complete.txt)
#             │       └─> html_to_text(html_content)
#             │               └─> _clean_whitespace(text)
#             │
#             └─> extract_10k_text_from_primary_html(primary.html) [Fallback]
#                     └─> html_to_text(html)
#                             └─> _clean_whitespace(text)
#
# OUTPUT:
#     {
#         '10k_2022': "UNITED STATES...",
#         '10k_2023': "UNITED STATES...",
#         '10k_2024': "UNITED STATES...",
#         '10q_latest': "UNITED STATES...",
#         'metadata': {...}
#     }
#
# ============================================================================

"""
EXAMPLE USAGE:

downloader = FilingDownloader(download_dir="./filings")

meta_texts = downloader.download_and_extract_for_chatbot(
    cik=1326801,
    current_year=2024,
    company_name="Meta"
)
"""

"""
Bsp.:
from download_filings import FilingDownloader

downloader = FilingDownloader(download_dir="./filings")
meta_texts = downloader.download_and_extract_for_chatbot(
    cik=1326801,
    current_year=2024,
    company_name="Meta"  # Optional, kann None sein
)
"""

class FilingDownloader:
    """
    Klasse zum Herunterladen von SEC 10-K Filings.
    """
    
    BASE_URL = "https://data.sec.gov/submissions"
    EDGAR_URL = "https://www.sec.gov/Archives/edgar/data"
    
    def __init__(self, download_dir: str = "./downloads", user_agent: Optional[str] = None):
        """
        Initialisiert den FilingDownloader.
        
        Args:
            download_dir: Verzeichnis zum Speichern der heruntergeladenen Filings
            user_agent: Optionaler User-Agent String (SEC empfiehlt: "Name email@example.com")
                       Falls None, wird ein Standard-Browser User-Agent verwendet
        """
        self.download_dir = Path(download_dir)
        self.download_dir.mkdir(parents=True, exist_ok=True)
        
        # SEC empfiehlt einen User-Agent mit Kontaktinformationen
        # z.B.: "YourCompany info@yourcompany.com"
        if user_agent is None:
            user_agent = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
        
        self.headers = {
            "User-Agent": user_agent
        }
        
    def get_submissions(self, cik: int) -> Dict:
        """
        Lädt die Submissions-Daten für eine CIK von der SEC API.
        
        Args:
            cik: Central Index Key der Firma
            
        Returns:
            Dict mit allen Submission-Daten
            
        Raises:
            requests.HTTPError: Wenn API-Request fehlschlägt
        """
        # CIK muss 10-stellig sein (mit führenden Nullen)
        cik_padded = str(cik).zfill(10)
        url = f"{self.BASE_URL}/CIK{cik_padded}.json"
        
        print(f"Lade Submissions für CIK {cik_padded}...")
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        
        return response.json()
    
    def get_all_submissions(self, cik: int) -> Dict:
        """
        Lädt ALLE Submissions-Daten inkl. älterer Filings aus Pagination.
        
        Args:
            cik: Central Index Key der Firma
            
        Returns:
            Dict mit allen Submission-Daten (recent + older files zusammengefügt)
        """
        # Lade Haupt-Submission
        submissions = self.get_submissions(cik)
        
        recent = submissions.get('filings', {}).get('recent', {})
        files = submissions.get('filings', {}).get('files', [])
        
        # Wenn es zusätzliche Files gibt, lade diese auch
        if files:
            print(f"Lade {len(files)} zusätzliche Filing-Archive...")
            
            for file_info in files:
                file_name = file_info.get('name')
                if not file_name:
                    continue
                
                # Files liegen direkt im submissions Ordner (nicht in CIK-Unterordner)
                url = f"{self.BASE_URL}/{file_name}"
                try:
                    response = requests.get(url, headers=self.headers)
                    response.raise_for_status()
                    older_data = response.json()
                    
                    # Ältere Files haben die Arrays direkt, nicht unter 'filings/recent'
                    # Füge die älteren Daten zu recent hinzu
                    for key in recent.keys():
                        if key in older_data and isinstance(older_data[key], list):
                            recent[key].extend(older_data[key])
                    
                    time.sleep(0.1)  # Rate limiting
                except Exception as e:
                    print(f"[WARN] Konnte {file_name} nicht laden: {e}")
        
        submissions['filings']['recent'] = recent
        return submissions
    
    def find_10k_filing(self, cik: int, year: int, include_amendments: bool = True, 
                       use_all_filings: bool = True) -> Optional[Dict]:
        """
        Findet ein 10-K Filing für ein bestimmtes Jahr.
        
        Args:
            cik: Central Index Key der Firma
            year: Geschäftsjahr (z.B. 2024)
            include_amendments: Ob 10-K/A (Amendments) auch berücksichtigt werden sollen
            use_all_filings: Wenn True, durchsucht auch ältere Filings (langsamer aber vollständig)
            
        Returns:
            Dict mit Filing-Informationen oder None wenn nicht gefunden
            Format: {
                'accessionNumber': '0001326801-25-000013',
                'filingDate': '2025-02-05',
                'primaryDocument': 'meta-20241231.htm',
                'form': '10-K',
                'fiscalYear': '2024'
            }
        """
        # Wenn ältere Filings durchsucht werden sollen, lade alle
        if use_all_filings:
            submissions = self.get_all_submissions(cik)
        else:
            submissions = self.get_submissions(cik)
        
        # Recent filings sind direkt im Hauptobjekt
        recent = submissions.get('filings', {}).get('recent', {})
        
        # Suche nach 10-K Filings für das angegebene Jahr
        forms = recent.get('form', [])
        filing_dates = recent.get('filingDate', [])
        accession_numbers = recent.get('accessionNumber', [])
        primary_docs = recent.get('primaryDocument', [])
        fiscal_years = recent.get('fiscalYear', [])  # SEC liefert direkt das Fiscal Year (wenn vorhanden)
        report_dates = recent.get('reportDate', [])   # Fallback: Ende des Fiscal Years
        
        for i, form in enumerate(forms):
            # Prüfe ob es ein 10-K Filing ist
            is_10k = (form == '10-K') or (include_amendments and form == '10-K/A')
            
            if is_10k:
                # Bestimme Fiscal Year:
                # 1. Aus fiscalYear Array (falls vorhanden)
                # 2. Aus reportDate Jahr (Fallback, da reportDate = Ende des FY)
                fiscal_year = None
                
                if fiscal_years and i < len(fiscal_years) and fiscal_years[i]:
                    fiscal_year = int(fiscal_years[i])
                elif report_dates and i < len(report_dates):
                    # Report Date ist Format YYYY-MM-DD (z.B. 2023-12-31)
                    # Jahr des Report Dates = Fiscal Year
                    fiscal_year = int(report_dates[i][:4])
                
                # Prüfe ob das Fiscal Year zum gewünschten Jahr passt
                if fiscal_year and fiscal_year == year:
                    return {
                        'accessionNumber': accession_numbers[i],
                        'filingDate': filing_dates[i],
                        'primaryDocument': primary_docs[i],
                        'form': form,
                        'fiscalYear': str(fiscal_year)
                    }
        
        print(f"Kein 10-K Filing für Fiscal Year {year} gefunden.")
        return None
    
    def download_filing_html(self, cik: int, accession_number: str, primary_doc: str, 
                            save_filename: Optional[str] = None) -> str:
        """
        Lädt das HTML-Dokument eines Filings herunter.
        
        Args:
            cik: Central Index Key
            accession_number: Accession Number (z.B. '0001326801-25-000013')
            primary_doc: Primary Document Name (z.B. 'meta-20241231.htm')
            save_filename: Optionaler Dateiname zum Speichern (default: primary_doc)
            
        Returns:
            Pfad zur gespeicherten Datei
            
        Raises:
            requests.HTTPError: Wenn Download fehlschlägt
        """
        # Entferne Bindestriche aus Accession Number
        accession_no_nodashes = accession_number.replace('-', '')
        
        # Baue URL zusammen
        url = f"{self.EDGAR_URL}/{cik}/{accession_no_nodashes}/{primary_doc}"
        
        print(f"Lade Filing von {url}...")
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        
        # Speichere Datei
        if save_filename is None:
            save_filename = f"CIK{str(cik).zfill(10)}_{accession_number.replace('-', '_')}_{primary_doc}"
        
        filepath = self.download_dir / save_filename
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(response.text)
        
        print(f"[OK] Filing gespeichert: {filepath}")
        return str(filepath)
    
    def download_complete_submission(self, cik: int, accession_number: str, 
                                    save_filename: Optional[str] = None) -> str:
        """
        Lädt die vollständige Submission als .txt Datei herunter.
        Dies enthält alle Dokumente des Filings in einem File.
        
        Args:
            cik: Central Index Key
            accession_number: Accession Number (z.B. '0001326801-25-000013')
            save_filename: Optionaler Dateiname zum Speichern
            
        Returns:
            Pfad zur gespeicherten Datei
            
        Raises:
            requests.HTTPError: Wenn Download fehlschlägt
        """
        # Entferne Bindestriche aus Accession Number
        accession_no_nodashes = accession_number.replace('-', '')
        
        # Complete Submission URL (enthält ALLE Dokumente)
        url = f"{self.EDGAR_URL}/{cik}/{accession_no_nodashes}/{accession_number}.txt"
        
        print(f"Lade vollständige Submission von {url}...")
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        
        # Speichere Datei
        if save_filename is None:
            save_filename = f"CIK{str(cik).zfill(10)}_{accession_number.replace('-', '_')}_complete.txt"
        
        filepath = self.download_dir / save_filename
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(response.text)
        
        print(f"[OK] Vollständige Submission gespeichert: {filepath}")
        return str(filepath)
    
    def download_filing_index(self, cik: int, accession_number: str,
                             save_filename: Optional[str] = None) -> str:
        """
        Lädt die Index-Seite eines Filings herunter.
        Diese listet alle enthaltenen Dokumente auf.
        
        Args:
            cik: Central Index Key
            accession_number: Accession Number
            save_filename: Optionaler Dateiname
            
        Returns:
            Pfad zur gespeicherten Index-Datei
        """
        accession_no_nodashes = accession_number.replace('-', '')
        
        # Index URL
        url = f"{self.EDGAR_URL}/{cik}/{accession_no_nodashes}/{accession_number}-index.html"
        
        print(f"Lade Filing Index von {url}...")
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        
        if save_filename is None:
            save_filename = f"CIK{str(cik).zfill(10)}_{accession_number.replace('-', '_')}_index.html"
        
        filepath = self.download_dir / save_filename
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(response.text)
        
        print(f"[OK] Filing Index gespeichert: {filepath}")
        return str(filepath)
    
    def download_10k_for_year(self, cik: int, year: int, company_name: Optional[str] = None, 
                             complete_submission: bool = True) -> Optional[Dict[str, str]]:
        """
        Findet und lädt das 10-K Filing für ein bestimmtes Jahr herunter.
        
        Args:
            cik: Central Index Key
            year: Geschäftsjahr
            company_name: Optionaler Firmenname für Dateinamen
            complete_submission: Wenn True, lädt vollständige .txt Submission (empfohlen)
                                Wenn False, lädt nur primaryDocument HTML
            
        Returns:
            Dict mit Pfaden: {'primary': '...', 'complete': '...', 'index': '...'} 
            oder None wenn nicht gefunden
        """
        print(f"\n{'='*60}")
        print(f"Suche 10-K Filing für CIK {cik}, Jahr {year}")
        print(f"{'='*60}\n")
        
        # Finde Filing
        filing_info = self.find_10k_filing(cik, year)
        
        if filing_info is None:
            return None
        
        print(f"[OK] Filing gefunden:")
        print(f"  Form: {filing_info['form']}")
        print(f"  Filing Date: {filing_info['filingDate']}")
        print(f"  Accession Number: {filing_info['accessionNumber']}")
        print(f"  Primary Document: {filing_info['primaryDocument']}\n")
        
        result = {}
        
        # Lade Primary Document (kann Cover Page sein)
        if company_name:
            primary_filename = f"{company_name.replace(' ', '_')}_{year}_10K_primary.html"
        else:
            primary_filename = f"CIK{str(cik).zfill(10)}_{year}_10K_primary.html"
        
        result['primary'] = self.download_filing_html(
            cik=cik,
            accession_number=filing_info['accessionNumber'],
            primary_doc=filing_info['primaryDocument'],
            save_filename=primary_filename
        )
        
        time.sleep(0.1)
        
        # Lade Complete Submission (alle Dokumente als .txt)
        if complete_submission:
            if company_name:
                complete_filename = f"{company_name.replace(' ', '_')}_{year}_10K_complete.txt"
            else:
                complete_filename = f"CIK{str(cik).zfill(10)}_{year}_10K_complete.txt"
            
            result['complete'] = self.download_complete_submission(
                cik=cik,
                accession_number=filing_info['accessionNumber'],
                save_filename=complete_filename
            )
            
            time.sleep(0.1)
            
            # Lade auch Index
            if company_name:
                index_filename = f"{company_name.replace(' ', '_')}_{year}_10K_index.html"
            else:
                index_filename = f"CIK{str(cik).zfill(10)}_{year}_10K_index.html"
            
            try:
                result['index'] = self.download_filing_index(
                    cik=cik,
                    accession_number=filing_info['accessionNumber'],
                    save_filename=index_filename
                )
            except requests.HTTPError as e:
                print(f"[WARN] Index nicht verfügbar: {e}")
                result['index'] = None
        
        # Kurze Pause um SEC Server nicht zu überlasten
        time.sleep(0.2)

        result['filing_date'] = filing_info['filingDate']
        result['fiscal_year'] = year
        
        return result
    
    def list_all_10k_filings(self, cik: int) -> List[Dict]:
        """
        Listet alle verfügbaren 10-K Filings für eine CIK auf.
        
        Args:
            cik: Central Index Key
            
        Returns:
            Liste von Dicts mit Filing-Informationen
        """
        submissions = self.get_submissions(cik)
        recent = submissions.get('filings', {}).get('recent', {})
        
        forms = recent.get('form', [])
        filing_dates = recent.get('filingDate', [])
        accession_numbers = recent.get('accessionNumber', [])
        primary_docs = recent.get('primaryDocument', [])
        
        filings_10k = []
        
        for i, form in enumerate(forms):
            if form in ['10-K', '10-K/A']:
                filings_10k.append({
                    'form': form,
                    'filingDate': filing_dates[i],
                    'accessionNumber': accession_numbers[i],
                    'primaryDocument': primary_docs[i]
                })
        
        return filings_10k
    
    def find_latest_10q_filing(self, cik: int) -> Optional[Dict]:
        """
        Findet das neueste 10-Q Filing (nach dem letzten 10-K).
        
        Args:
            cik: Central Index Key
            
        Returns:
            Dict mit Filing-Informationen oder None wenn nicht gefunden
        """
        submissions = self.get_submissions(cik)
        recent = submissions.get('filings', {}).get('recent', {})
        
        forms = recent.get('form', [])
        filing_dates = recent.get('filingDate', [])
        accession_numbers = recent.get('accessionNumber', [])
        primary_docs = recent.get('primaryDocument', [])
        
        # Finde zuerst das neueste 10-K
        latest_10k_date = None
        for i, form in enumerate(forms):
            if form in ['10-K', '10-K/A']:
                latest_10k_date = filing_dates[i]
                break
        
        # Finde neuestes 10-Q nach dem letzten 10-K
        for i, form in enumerate(forms):
            if form in ['10-Q', '10-Q/A']:
                filing_date = filing_dates[i]
                # Wenn es nach dem letzten 10-K kommt (oder kein 10-K existiert)
                if latest_10k_date is None or filing_date > latest_10k_date:
                    return {
                        'accessionNumber': accession_numbers[i],
                        'filingDate': filing_date,
                        'primaryDocument': primary_docs[i],
                        'form': form
                    }
        
        print("Kein 10-Q Filing nach dem letzten 10-K gefunden.")
        return None

    def find_recent_10q_filings(self, cik: int, max_filings: int = 4, after_date: Optional[str] = None) -> List[Dict]:
        """
        Findet die neuesten 10-Q Filings nach dem letzten 10-K.

        Args:
            cik: Central Index Key
            max_filings: Maximale Anzahl zurückzugebender 10-Q Filings
            after_date: Optionales Referenzdatum; falls gesetzt, nur 10-Q nach diesem Datum

        Returns:
            Liste von Filing-Informationen (neueste zuerst)
        """
        submissions = self.get_submissions(cik)
        recent = submissions.get('filings', {}).get('recent', {})

        forms = recent.get('form', [])
        filing_dates = recent.get('filingDate', [])
        accession_numbers = recent.get('accessionNumber', [])
        primary_docs = recent.get('primaryDocument', [])

        latest_10k_date = after_date
        if latest_10k_date is None:
            for i, form in enumerate(forms):
                if form in ['10-K', '10-K/A']:
                    latest_10k_date = filing_dates[i]
                    break

        tenq_candidates = []
        for i, form in enumerate(forms):
            if form in ['10-Q', '10-Q/A']:
                filing_date = filing_dates[i]
                if latest_10k_date is None or filing_date > latest_10k_date:
                    tenq_candidates.append({
                        'accessionNumber': accession_numbers[i],
                        'filingDate': filing_date,
                        'primaryDocument': primary_docs[i],
                        'form': form
                    })

        tenq_candidates.sort(key=lambda x: x['filingDate'], reverse=True)
        return tenq_candidates[:max_filings]
    
    def download_multiple_years(self, cik: int, years: List[int], company_name: Optional[str] = None) -> List[Dict[str, str]]:
        """
        Lädt 10-K Filings für mehrere Jahre herunter.
        
        Args:
            cik: Central Index Key
            years: Liste von Geschäftsjahren (z.B. [2022, 2023, 2024])
            company_name: Optionaler Firmenname für Dateinamen
            
        Returns:
            Liste von Dicts mit Pfaden zu gespeicherten Dateien
        """
        results = []
        
        for year in years:
            result = self.download_10k_for_year(cik, year, company_name)
            if result:
                results.append(result)
            time.sleep(0.2)  # Rate limiting
        
        return results
    
    def download_latest_10q(self, cik: int, company_name: Optional[str] = None, 
                          complete_submission: bool = True) -> Optional[Dict[str, str]]:
        """
        Lädt das neueste 10-Q Filing herunter.
        
        Args:
            cik: Central Index Key
            company_name: Optionaler Firmenname für Dateinamen
            complete_submission: Wenn True, lädt vollständige .txt Submission
            
        Returns:
            Dict mit Pfaden oder None wenn nicht gefunden
        """
        print(f"\n{'='*60}")
        print(f"Suche neuestes 10-Q Filing für CIK {cik}")
        print(f"{'='*60}\n")
        
        filing_info = self.find_latest_10q_filing(cik)
        
        if filing_info is None:
            return None
        
        print(f"[OK] Filing gefunden:")
        print(f"  Form: {filing_info['form']}")
        print(f"  Filing Date: {filing_info['filingDate']}")
        print(f"  Accession Number: {filing_info['accessionNumber']}")
        print(f"  Primary Document: {filing_info['primaryDocument']}\n")
        
        result = {}
        
        # Erstelle aussagekräftigen Dateinamen
        if company_name:
            primary_filename = f"{company_name.replace(' ', '_')}_{filing_info['filingDate']}_10Q_primary.html"
        else:
            primary_filename = f"CIK{str(cik).zfill(10)}_{filing_info['filingDate']}_10Q_primary.html"
        
        result['primary'] = self.download_filing_html(
            cik=cik,
            accession_number=filing_info['accessionNumber'],
            primary_doc=filing_info['primaryDocument'],
            save_filename=primary_filename
        )
        
        time.sleep(0.1)
        
        if complete_submission:
            if company_name:
                complete_filename = f"{company_name.replace(' ', '_')}_{filing_info['filingDate']}_10Q_complete.txt"
            else:
                complete_filename = f"CIK{str(cik).zfill(10)}_{filing_info['filingDate']}_10Q_complete.txt"
            
            result['complete'] = self.download_complete_submission(
                cik=cik,
                accession_number=filing_info['accessionNumber'],
                save_filename=complete_filename
            )
            
            time.sleep(0.1)
            
            if company_name:
                index_filename = f"{company_name.replace(' ', '_')}_{filing_info['filingDate']}_10Q_index.html"
            else:
                index_filename = f"CIK{str(cik).zfill(10)}_{filing_info['filingDate']}_10Q_index.html"
            
            try:
                result['index'] = self.download_filing_index(
                    cik=cik,
                    accession_number=filing_info['accessionNumber'],
                    save_filename=index_filename
                )
            except requests.HTTPError as e:
                print(f"[WARN] Index nicht verfügbar: {e}")
                result['index'] = None
        
        return result

    def download_recent_10q(self, cik: int, company_name: Optional[str] = None,
                            complete_submission: bool = True, max_filings: int = 4,
                            after_date: Optional[str] = None) -> List[Dict[str, str]]:
        """
        Lädt die neuesten 10-Q Filings (mehrere) herunter.

        Args:
            cik: Central Index Key
            company_name: Optionaler Firmenname für Dateinamen
            complete_submission: Wenn True, lädt vollständige .txt Submission
            max_filings: Maximale Anzahl herunterzuladender 10-Q Filings
            after_date: Optionales Referenzdatum; falls gesetzt, nur 10-Q nach diesem Datum

        Returns:
            Liste von Dicts mit Pfaden (neueste zuerst)
        """
        print(f"\n{'='*60}")
        print(f"Suche bis zu {max_filings} neueste 10-Q Filings für CIK {cik}")
        print(f"{'='*60}\n")

        filing_infos = self.find_recent_10q_filings(cik=cik, max_filings=max_filings, after_date=after_date)
        if not filing_infos:
            print("Keine passenden 10-Q Filings gefunden.")
            return []

        results = []
        for filing_info in filing_infos:
            print(f"[OK] Filing gefunden:")
            print(f"  Form: {filing_info['form']}")
            print(f"  Filing Date: {filing_info['filingDate']}")
            print(f"  Accession Number: {filing_info['accessionNumber']}")
            print(f"  Primary Document: {filing_info['primaryDocument']}\n")

            result = {}

            if company_name:
                primary_filename = f"{company_name.replace(' ', '_')}_{filing_info['filingDate']}_10Q_primary.html"
            else:
                primary_filename = f"CIK{str(cik).zfill(10)}_{filing_info['filingDate']}_10Q_primary.html"

            result['primary'] = self.download_filing_html(
                cik=cik,
                accession_number=filing_info['accessionNumber'],
                primary_doc=filing_info['primaryDocument'],
                save_filename=primary_filename
            )
            result['filing_date'] = filing_info['filingDate']
            filing_date = filing_info.get('filingDate')
            result['fiscal_year'] = int(filing_date[:4]) if filing_date and len(filing_date) >= 4 else None

            time.sleep(0.1)

            if complete_submission:
                if company_name:
                    complete_filename = f"{company_name.replace(' ', '_')}_{filing_info['filingDate']}_10Q_complete.txt"
                else:
                    complete_filename = f"CIK{str(cik).zfill(10)}_{filing_info['filingDate']}_10Q_complete.txt"

                result['complete'] = self.download_complete_submission(
                    cik=cik,
                    accession_number=filing_info['accessionNumber'],
                    save_filename=complete_filename
                )

                time.sleep(0.1)

                if company_name:
                    index_filename = f"{company_name.replace(' ', '_')}_{filing_info['filingDate']}_10Q_index.html"
                else:
                    index_filename = f"CIK{str(cik).zfill(10)}_{filing_info['filingDate']}_10Q_index.html"

                try:
                    result['index'] = self.download_filing_index(
                        cik=cik,
                        accession_number=filing_info['accessionNumber'],
                        save_filename=index_filename
                    )
                except requests.HTTPError as e:
                    print(f"[WARN] Index nicht verfügbar: {e}")
                    result['index'] = None

            result['filing_date'] = filing_info['filingDate']
            results.append(result)
            time.sleep(0.2)

        return results
    
    def download_complete_package(self, cik: int, current_year: int, company_name: Optional[str] = None) -> Dict[str, any]:
        """
        Lädt ein komplettes Paket: 10-K für 3 Jahre + neuestes 10-Q.
        Jedes Filing enthält: primary HTML, complete .txt Submission, und index HTML.
        
        Args:
            cik: Central Index Key
            current_year: Aktuelles/neuestes Jahr für 10-K
            company_name: Optionaler Firmenname
            
        Returns:
            Dict: {'10k_files': [{'primary': '...', 'complete': '...', 'index': '...'}], '10q_file': {...}}
        """
        print(f"\n{'='*70}")
        print(f"KOMPLETTER DOWNLOAD FÜR {company_name or f'CIK {cik}'}")
        print(f"{'='*70}\n")
        
        # Lade 10-K für 3 Jahre
        years = [current_year - 2, current_year - 1, current_year]
        print(f"Lade 10-K Reports für Jahre: {years}")
        tenk_files = self.download_multiple_years(cik, years, company_name)
        
        # Referenzdatum: Filing-Date des gewählten current_year 10-K
        reference_10k_date = None
        for filing in tenk_files:
            if filing.get('fiscal_year') == current_year:
                reference_10k_date = filing.get('filing_date')
                break

        # Lade neueste 10-Qs (mehrere)
        print(f"\nLade neueste 10-Q Reports...")
        tenq_files = self.download_recent_10q(cik, company_name, max_filings=4, after_date=reference_10k_date)
        tenq_file = tenq_files[0] if tenq_files else None
        
        print(f"\n{'='*70}")
        print(f"DOWNLOAD ABGESCHLOSSEN")
        print(f"{'='*70}")
        print(f"10-K Reports: {len(tenk_files)} Dateien")
        print(f"10-Q Reports: {len(tenq_files)} Dateien")
        print(f"\nJedes Filing enthält:")
        print(f"  • primary HTML (primaryDocument, kann Cover Page sein)")
        print(f"  • complete .txt (vollständige Submission mit allen Dokumenten)")
        print(f"  • index HTML (Übersicht aller Dokumente)")
        
        return {
            '10k_files': tenk_files,
            '10q_file': tenq_file,
            '10q_files': tenq_files
        }
    
    def download_and_extract_for_chatbot(self, cik: int, current_year: int, 
                                        company_name: Optional[str] = None,
                                        min_chars: int = 20000) -> Dict[str, str]:
        """
        All-in-One Methode für Chatbot: Lädt 3 Jahre 10-K + neuesten 10-Q
        und extrahiert sofort die bereinigten Texte.
        
        Args:
            cik: Central Index Key
            current_year: Aktuelles/neuestes Jahr für 10-K
            company_name: Optionaler Firmenname
            min_chars: Minimale Textlänge für Validierung
            
        Returns:
            Dict mit bereinigten Texten:
            {
                '10k_2022': "UNITED STATES...",
                '10k_2023': "UNITED STATES...",
                '10k_2024': "UNITED STATES...",
                '10q_latest': "UNITED STATES...",
                '10q_YYYY-MM-DD': "UNITED STATES...",
                'metadata': {'sources': {...}, 'company': '...', 'years': [...]}
            }
        """
        print(f"\n{'='*70}")
        print(f"CHATBOT-DOWNLOAD & TEXT-EXTRAKTION FÜR {company_name or f'CIK {cik}'}")
        print(f"{'='*70}\n")
        
        # Lade alle Filings
        package = self.download_complete_package(cik, current_year, company_name)
        
        print(f"\n{'='*70}")
        print(f"TEXT-EXTRAKTION")
        print(f"{'='*70}\n")
        
        texts = {}
        metadata = {
            'sources': {},
            'company': company_name or f'CIK_{cik}',
            'years': [current_year - 2, current_year - 1, current_year],
            'has_10q': package['10q_file'] is not None,
            '10q_count': len(package.get('10q_files', [])),
            '10q_dates': [f.get('filing_date') for f in package.get('10q_files', []) if f.get('filing_date')]
        }
        
        # Extrahiere 10-K Texte
        years = [current_year - 2, current_year - 1, current_year]
        for i, (year, filing) in enumerate(zip(years, package['10k_files'])):
            try:
                print(f"Extrahiere 10-K {year}...")
                text, source = extract_clean_10k_text(filing, min_chars)
                texts[f'10k_{year}'] = text
                metadata['sources'][f'10k_{year}'] = source
                print(f"  [OK] {len(text):,} Zeichen aus {source}")
            except ValueError as e:
                print(f"  [ERROR] Fehler: {e}")
                texts[f'10k_{year}'] = None
                metadata['sources'][f'10k_{year}'] = 'failed'
        
        # Extrahiere 10-Q Texte (mehrere)
        tenq_files = package.get('10q_files') or ([] if package['10q_file'] is None else [package['10q_file']])
        if tenq_files:
            latest_set = False
            for idx, tenq_filing in enumerate(tenq_files):
                filing_date = tenq_filing.get('filing_date')
                key = f"10q_{filing_date}" if filing_date else f"10q_{idx + 1}"
                try:
                    print(f"\nExtrahiere 10-Q ({filing_date or idx + 1})...")
                    text, source = extract_clean_10k_text(tenq_filing, min_chars=10000)  # 10-Q ist kürzer
                    texts[key] = text
                    metadata['sources'][key] = source
                    if not latest_set:
                        texts['10q_latest'] = text
                        metadata['sources']['10q_latest'] = source
                        latest_set = True
                    print(f"  [OK] {len(text):,} Zeichen aus {source}")
                except ValueError as e:
                    print(f"  [ERROR] Fehler: {e}")
                    texts[key] = None
                    metadata['sources'][key] = 'failed'
            if not latest_set:
                texts['10q_latest'] = None
                metadata['sources']['10q_latest'] = 'failed'
        else:
            texts['10q_latest'] = None
            metadata['sources']['10q_latest'] = 'not_available'
        
        texts['metadata'] = metadata
        
        print(f"\n{'='*70}")
        print(f"EXTRAKTION ABGESCHLOSSEN")
        print(f"{'='*70}")
        print(f"Erfolgreich extrahiert:")
        for key, text in texts.items():
            if key != 'metadata' and text:
                print(f"  [OK] {key}: {len(text):,} Zeichen, {len(text.split()):,} Wörter")
        
        return texts




# _clean_whitespace(s: str) -> str

# normalisiert Text
# ersetzt mehrere Leerzeichen/Zeilenumbrüche durch ein einzelnes Leerzeichen
# macht den Text RAG-tauglich (gleichmäßige Tokenisierung)

# html_to_text(html: str) -> str

# konvertiert HTML → reinen Text
# entfernt irrelevante Tags (script, style, noscript)
# extrahiert sichtbaren Text mit BeautifulSoup
# nutzt _clean_whitespace zur Nachbearbeitung

# extract_10k_text_from_complete_submission(path_txt: str) -> str

# liest die vollständige .txt Submission
# zerlegt sie in einzelne <DOCUMENT>-Blöcke
# identifiziert den Haupt-10-K-Block (<TYPE>10-K oder 10K)
# extrahiert dessen <TEXT>-Inhalt
# erkennt automatisch, ob der Inhalt HTML ist
# wählt den längsten sinnvollen Text als Hauptdokument

# extract_10k_text_from_primary_html(path_html: str) -> str
# liest das Primary HTML des Filings
# konvertiert es vollständig in reinen Text
# dient als Fallback, falls die complete submission unbrauchbar ist

# extract_clean_10k_text(filing_paths: dict, min_chars: int) -> (text, source)

# zentrale Orchestrierungsfunktion
# versucht zuerst:

# complete .txt → höchste Robustheit

# primary HTML → Fallback

# prüft Textlänge als Qualitätskontrolle

# gibt zurück:

# text: bereinigter 10-K-Text

# source: "complete_txt" oder "primary_html"




import re
from pathlib import Path
from bs4 import BeautifulSoup

def _clean_whitespace(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(" ")
    return _clean_whitespace(text)

def extract_10k_text_from_complete_submission(path_txt: str) -> str:
    raw = Path(path_txt).read_text(encoding="utf-8", errors="ignore")

    # Split documents
    docs = raw.split("<DOCUMENT>")
    best_text = ""

    for d in docs[1:]:
        # Find TYPE
        m_type = re.search(r"<TYPE>\s*([^\s<]+)", d)
        doc_type = (m_type.group(1).strip().upper() if m_type else "")

        # Heuristic: prefer 10-K main doc
        if doc_type in {"10-K", "10K"}:
            m_text = re.search(r"<TEXT>(.*)</TEXT>", d, flags=re.DOTALL | re.IGNORECASE)
            if not m_text:
                continue
            content = m_text.group(1)

            # content may be html
            if "<html" in content.lower() or "<table" in content.lower():
                cand = html_to_text(content)
            else:
                cand = _clean_whitespace(content)

            # pick the longest candidate
            if len(cand) > len(best_text):
                best_text = cand

    return best_text

def extract_10k_text_from_primary_html(path_html: str) -> str:
    html = Path(path_html).read_text(encoding="utf-8", errors="ignore")
    return html_to_text(html)

def extract_clean_10k_text(filing_paths: dict, min_chars: int = 20000) -> tuple[str, str]:
    """
    filing_paths: dict wie du es zurückgibst: {'primary':..., 'complete':..., 'index':...}
    returns: (text, source) wobei source in {"complete_txt","primary_html"}
    """
    text = ""
    source = ""

    complete_path = filing_paths.get("complete")
    primary_path = filing_paths.get("primary")

    if complete_path:
        text = extract_10k_text_from_complete_submission(complete_path)
        source = "complete_txt"

    if len(text) < min_chars and primary_path:
        text2 = extract_10k_text_from_primary_html(primary_path)
        if len(text2) > len(text):
            text = text2
            source = "primary_html"

    if len(text) < min_chars:
        raise ValueError(f"Extracted text too short ({len(text)} chars). Likely not the full 10-K.")

    return text, source




In [21]:
# Konfiguration
BIG_TECH = {
    "Apple": 320193,
    "Microsoft": 789019,
    "Alphabet": 1652044,
    "Amazon": 1018724,
    "Meta": 1326801,
    "NVIDIA": 1045810,
    "Tesla": 1318605,
}



# Email für EDGAR API, damit die Anfragen nicht blockiert werden
SEC_USER_AGENT = "EdinDzeko@Goal.com"
main_api.SEC_HEADERS["User-Agent"] = SEC_USER_AGENT



metrics_df = load_three_metrics(BIG_TECH["Meta"], start_year=2001, end_year=2024)
metrics_df

,year,metric,value,form,mode
0,2001,diluted_earnings,NaN,10-K,direct
1,2001,capex,NaN,10-K,normalized_abs
2,2001,fcf,NaN,10-K,direct_from_companyfacts
3,2002,diluted_earnings,NaN,10-K,direct
4,2002,capex,NaN,10-K,normalized_abs
...,...,...,...,...,...
67,2023,capex,2.726600e+10,10-K,normalized_abs
68,2023,fcf,4.384700e+10,10-K,direct_from_companyfacts
69,2024,diluted_earnings,2.386000e+01,10-K,direct
70,2024,capex,3.725600e+10,10-K,normalized_abs


## LLM-Funktion 

Default: Cloud-Free-Tier ueber OpenRouter. Erstellen Sie einen kostenlosen API-Key dafür im Netz. Alternativ: Eigener Open-AI-Key für leistungsfährigere Modelle.

In [22]:
import os
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
if not openrouter_api_key:
    print("Bitte setzen Sie die Variable OPENROUTER_API_KEY mit Ihrem OpenRouter API-Schlüssel.")
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API Key eingeben (wird nicht angezeigt): ")


In [23]:


def ask_llm(question: str, context: str = "") -> str:
    """Cloud-Free-Tier ueber OpenRouter."""
    api_key = os.getenv("OPENROUTER_API_KEY", "")
    if not api_key:
        return "Kein OPENROUTER_API_KEY gesetzt."

    url = _build_openrouter_url()
    model = os.getenv("OPENROUTER_MODEL", "openrouter/auto")

    prompt = f"Kontext:\n{context}\n\nFrage: {question}\nAntworte kurz und klar."

    try:
        r = requests.post(
            url,
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
                # Empfohlen von OpenRouter, hilft bei einigen Setups.
                "HTTP-Referer": os.getenv("OPENROUTER_SITE_URL", "http://localhost"),
                "X-Title": os.getenv("OPENROUTER_APP_NAME", "SEC Teaching Notebook"),
            },
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0,
            },
            timeout=120,
        )

        if r.status_code >= 400:
            # Detailierte Antwort hilft bei 404/401-Diagnose direkt im Notebook.
            body = r.text[:500]
            if r.status_code == 404:
                return (
                    f"LLM-API Fehler 404: Endpoint oder Modell nicht gefunden. "
                    f"URL={url}, model={model}, body={body}"
                )
            return f"LLM-API Fehler {r.status_code}: {body}"

        data = r.json()
        return data.get("choices", [{}])[0].get("message", {}).get("content", "Keine Antwort erhalten.")
    except Exception as e:
        return f"LLM-API Fehler: {e}"

## RAG für letzte 2x 10-K Filings

Dieser Abschnitt baut den Retrieval-Kontext aus den letzten zwei verfuegbaren 10-K Filings direkt ueber die SEC-API auf. 

1. Frage wird in Tokens zerlegt (q_tokens), z. B. Wörter wie „cash“, „flow“, „meta“.
2. Die letzten 2 passenden 10-K Filings werden über SEC geholt.
3. Aus jedem Filing wird der Text extrahiert.
4. Der Text wird in kleinere Abschnitte (Chunks) geteilt.
5. Jeder Chunk wird ebenfalls in Tokens zerlegt (c_tokens).
6. Pro Chunk wird ein einfacher Relevanz-Score berechnet:
```python
len(q_tokens.intersection(c_tokens))
```
Es wird gezählt, wie viele verschiedene Wörter in Frage und Chunk gleichzeitig vorkommen. Das ist eine sehr einfache Relevanz-Quantifizierung. Alternativen würden z.B. die Distanz zwischen dem Durchschnitt aus Embedding-Vektoren über alle Token in Query und einzelnen Chunks berechnen.

7. Die Top-k Chunks mit dem höchsten Score werden ausgewählt.
Diese Chunks werden als RAG-Kontext an das LLM gegeben.





In [27]:
import re
import requests
from bs4 import BeautifulSoup



def _normalize_text(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", (s or "").lower())


def _tokenize_for_rag(text: str) -> set[str]:
    tokens = re.findall(r"[a-zA-Z0-9]{3,}", (text or "").lower())
    stopwords = {
        "the", "and", "for", "with", "from", "that", "this", "are", "was", "were", "have", "has",
        "ein", "eine", "und", "der", "die", "das", "ist", "mit", "von", "auf", "wie", "sind",
        "what", "when", "where", "which", "uber", "into", "also", "their", "its", "our",
    }
    return {t for t in tokens if t not in stopwords}


def _clean_whitespace(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()


def _html_to_text(html: str) -> str:
    soup = BeautifulSoup(html or "", "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    return _clean_whitespace(soup.get_text(" "))


def _extract_10k_text_from_submission_raw(raw_submission: str) -> str:
    docs = (raw_submission or "").split("<DOCUMENT>")
    best_text = ""

    for d in docs[1:]:
        m_type = re.search(r"<TYPE>\s*([^\s<]+)", d)
        doc_type = (m_type.group(1).strip().upper() if m_type else "")

        if doc_type in {"10-K", "10K"}:
            m_text = re.search(r"<TEXT>(.*)</TEXT>", d, flags=re.DOTALL | re.IGNORECASE)
            if not m_text:
                continue
            content = m_text.group(1)

            if "<html" in content.lower() or "<table" in content.lower():
                cand = _html_to_text(content)
            else:
                cand = _clean_whitespace(content)

            if len(cand) > len(best_text):
                best_text = cand

    return best_text


def _fetch_complete_submission_text(cik: int, accession_number: str, headers: dict) -> str:
    accession_no_nodashes = accession_number.replace("-", "")
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_nodashes}/{accession_number}.txt"
    r = requests.get(url, headers=headers, timeout=120)
    r.raise_for_status()
    return r.text


def _find_latest_two_10k_filings_api(company_name: str) -> list[dict]:
    cik = BIG_TECH[company_name]
    downloader = FilingDownloader(download_dir="./downloads", user_agent=SEC_USER_AGENT)

    submissions = downloader.get_all_submissions(cik)
    recent = submissions.get("filings", {}).get("recent", {})

    forms = recent.get("form", [])
    filing_dates = recent.get("filingDate", [])
    accession_numbers = recent.get("accessionNumber", [])
    fiscal_years = recent.get("fiscalYear", [])
    report_dates = recent.get("reportDate", [])

    candidates = []
    for i, form in enumerate(forms):
        if form not in {"10-K", "10-K/A"}:
            continue

        fy = None
        if fiscal_years and i < len(fiscal_years) and fiscal_years[i]:
            fy = int(fiscal_years[i])
        elif report_dates and i < len(report_dates) and report_dates[i]:
            fy = int(str(report_dates[i])[:4])

        candidates.append({
            "fiscal_year": fy if fy is not None else 0,
            "filing_date": filing_dates[i] if i < len(filing_dates) else "",
            "accession_number": accession_numbers[i] if i < len(accession_numbers) else "",
            "cik": cik,
            "headers": downloader.headers,
        })

    candidates = [c for c in candidates if c["accession_number"]]

    unique_by_year = {}
    for c in sorted(candidates, key=lambda x: x["filing_date"], reverse=True):
        fy = c["fiscal_year"]
        if fy not in unique_by_year:
            unique_by_year[fy] = c

    latest = sorted(unique_by_year.values(), key=lambda x: x["fiscal_year"], reverse=True)[:2]
    return latest


def _chunk_text(text: str, chunk_size: int = 1400, overlap: int = 250) -> list[str]:
    text = (text or "").strip()
    if not text:
        return []

    chunks = []
    start = 0
    n = len(text)
    step = max(1, chunk_size - overlap)
    while start < n:
        end = min(n, start + chunk_size)
        chunks.append(text[start:end])
        if end >= n:
            break
        start += step
    return chunks


def build_10k_rag_context(company_name: str, question: str, top_k: int = 4) -> tuple[str, list[str]]:
    filings = _find_latest_two_10k_filings_api(company_name)
    if not filings:
        return "", []

    q_tokens = _tokenize_for_rag(question)
    scored_chunks = []
    used_sources = []

    for filing in filings:
        try:
            raw = _fetch_complete_submission_text(
                cik=filing["cik"],
                accession_number=filing["accession_number"],
                headers=filing["headers"],
            )
            txt = _extract_10k_text_from_submission_raw(raw)
        except Exception:
            continue

        if not txt:
            continue

        src = f"FY{filing['fiscal_year']} ({filing['filing_date']})"
        used_sources.append(src)

        for ch in _chunk_text(txt):
            c_tokens = _tokenize_for_rag(ch)
            score = len(q_tokens.intersection(c_tokens)) if q_tokens else 0
            scored_chunks.append((score, src, ch))

    if not scored_chunks:
        return "", used_sources

    scored_chunks.sort(key=lambda x: x[0], reverse=True)
    selected = scored_chunks[:top_k]

    parts = []
    for i, (_, src, ch) in enumerate(selected, start=1):
        snippet = ch[:1200].replace("\n", " ").strip()
        parts.append(f"[{i}] Quelle={src} | Auszug: {snippet}")

    return "\n\n".join(parts), used_sources

## Query an LLM mit RAG und Übergabe konkreter Daten

In [30]:
company_name = "Meta"
# Query des Nutzers
question = "Wie haben sich aus welchen Gründen die CAPEX entwickelt?"

# Abfrage der Rohdaten und Berechnen der Bewertungsmultuples
df = load_three_metrics(BIG_TECH[company_name], 2020, 2024)
metrics_ctx = df.pivot_table(index="year", columns="metric", values="value", aggfunc="first").reset_index().to_string(index=False)

# Abfrage der Unternehmensreports und Berechnen der RAG Scores
rag_ctx, sources = build_10k_rag_context(company_name=company_name, question=question)

# Aufbauen des zusätzlichen Kontextes (neben der Query) für die LLM-Abfrage
combined_ctx = f"Unternehmen: {company_name}\n\nMetriken:\n{metrics_ctx}\n\nRAG Kontext:\n{rag_ctx}"

# Anfrage an das LLM mit dem gesamten (kombinierten) Kontext
answer = ask_llm(question, context=combined_ctx)
print(answer)

Lade Submissions für CIK 0001326801...
Lade 2 zusätzliche Filing-Archive...
Die CAPEX von Meta sind von 2020 bis 2024 stark gestiegen, mit einem deutlichen Sprung im Jahr 2022. Dies ist wahrscheinlich auf die massiven Investitionen in das Metaverse und die Infrastruktur zurückzuführen.


## Mini Dash Ansicht

In [ ]:
app = Dash(__name__)

app.layout = html.Div([
    html.H3("SEC Teaching Mini App"),
    html.Div([
        dcc.Dropdown(
            id="company_dd",
            options=[{"label": name, "value": name} for name in BIG_TECH.keys()],
            value= list(BIG_TECH.keys())[0],
            clearable=False,
            style={"width": "260px"},
        ),
        dcc.Dropdown(
            id="metric_dd",
            options=[
                {"label": "Diluted Earnings", "value": "diluted_earnings"},
                {"label": "CAPEX", "value": "capex"},
                {"label": "FCF", "value": "fcf"},
            ],
            value="fcf",
            clearable=False,
            style={"width": "220px"},
        ),
    ], style={"display": "flex", "gap": "12px", "marginBottom": "10px"}),
    dcc.Graph(id="line_fig"),
    html.Div(id="simple_label", style={"marginTop": "8px", "marginBottom": "12px"}),
    dcc.Textarea(id="chat_input", placeholder="Frage z. B.: Wie ist der FCF?", style={"width": "100%", "height": "80px"}),
    html.Button("Senden", id="send_btn", n_clicks=0, style={"marginTop": "8px"}),
    html.Div(id="chat_output", style={"marginTop": "10px", "padding": "10px", "backgroundColor": "#f6f8fa"}),
])

@app.callback(
    Output("line_fig", "figure"),
    Output("simple_label", "children"),
    Input("company_dd", "value"),
    Input("metric_dd", "value"),
)
def update_chart(company_name, metric):
    df = load_three_metrics(BIG_TECH[company_name], 2020, 2024)
    d = df[df["metric"] == metric].sort_values("year")
    fig = px.line(d, x="year", y="value", markers=True, title=f"{company_name}: {metric} pro Jahr")
    latest = d.iloc[-1]["value"] if not d.empty else None
    label = f"{company_name}: kein aktueller Wert" if latest is None or pd.isna(latest) else f"{company_name}: {metric} aktuell {'positiv' if latest > 0 else 'nicht positiv'}"
    return fig, label

@app.callback(
    Output("chat_output", "children"),
    Input("send_btn", "n_clicks"),
    State("chat_input", "value"),
    State("company_dd", "value"),
)
def update_chat(n_clicks, question, company_name):
    if not n_clicks:
        return "Noch keine Frage gestellt."

    df = load_three_metrics(BIG_TECH[company_name], 2020, 2024)
    metrics_ctx = df.pivot_table(index="year", columns="metric", values="value", aggfunc="first").reset_index().to_string(index=False)

    rag_ctx, sources = build_10k_rag_context(company_name=company_name, question=question or "")
    combined_ctx = f"Unternehmen: {company_name}\n\nMetriken:\n{metrics_ctx}"
    if rag_ctx:
        combined_ctx += f"\n\nRAG (letzte 2x 10-K):\n{rag_ctx}"

    try:
        answer = ask_llm(question or "", context=combined_ctx)
    except Exception as e:
        answer = f"LLM-Fehler: {e}"

    if sources:
        src_text = "Quellen (RAG): " + ", ".join(sources)
    else:
        src_text = "Quellen (RAG): keine 10-K Dateien gefunden"

    return [html.B("Antwort: "), answer, html.Br(), html.Small(src_text)]

In [29]:
# App im Notebook starten
app.run(jupyter_mode="inline", debug=False, port=8056)

Lade Submissions für CIK 0000320193...
Lade 1 zusätzliche Filing-Archive...
